In [ ]:
import matplotlib.pyplot as plt
from pathlib import Path
import pandas as pd
import numpy as np
import sys
import os

# --- Ajuste de sys.path para tu módulo RRAM ---
base_path = Path("c:/Users/Usuario/Documents/GitHub/RRAM_Simulation")
if str(base_path) not in sys.path:
    sys.path.append(str(base_path))

# Ahora importa el módulo
from RRAM.Representate import config_ax, setup_plt  # noqa: E402
from RRAM import io_manager as io  # noqa: E402

# --- Configuración de la figura ---
# Inicializa el estilo global
setup_plt(plt, latex=True, scaling=2)

In [ ]:
# --- Carpeta destino de todas las figuras ---
figures_dir = base_path / "Results" / "Media_set"
figures_dir.mkdir(parents=True, exist_ok=True)

# --- Listar carpetas de simulación bajo Results/ ---
results_dir = base_path / "Results"
sim_dirs = sorted(
    d for d in results_dir.iterdir()
    if d.is_dir() and d.name.startswith("simulation_")
)
print(f"🔄 Encontradas {len(sim_dirs)} simulaciones: {[d.name for d in sim_dirs]}")

In [ ]:
# Diccionario para almacenar la columna de Tiempo y las columnas de Intensidad de cada simulación
all_data_for_df = {}

for sim_dir in sim_dirs:
    num = sim_dir.name.split("_")[1]
    print(f"\n▶️ Procesando simulación {num} …")
    print(f"   Ruta: {sim_dir}")
    # 1) Construir rutas de datos
    paths = {
        "csv_set": sim_dir / "set"   / f"Resultados_set_{num}.csv",
        "csv_reset": sim_dir / "reset"   / f"Resultados_reset_{num}.csv",
    }
    missing = [k for k,p in paths.items() if not p.exists()]
    if missing:
        print(f"⚠️ Sim {num}: faltan {missing}, omitiendo.")
        continue
    
    # 2) Cargar datos
    data = io.leer_csv(str(paths["csv_set"]))

    # La columna de tiempo es la misma para todas las simulaciones.
    # La añadimos solo una vez, preferiblemente en la primera iteración o antes.
    if "Tiempo [s]" not in all_data_for_df:
        all_data_for_df["Tiempo [s]"] = data['Voltaje [V]']
        
    # Añadimos la columna de intensidad de la simulación actual con un nombre único
    all_data_for_df[f"Intensidad_Sim_{num} [A]"] = data["Intensidad [A]"]

# 4) Construir el DataFrame final a partir del diccionario
df_all_simulations = pd.DataFrame(all_data_for_df)

# print(f"\n📊 DataFrame de todas las simulaciones:\n{df_all_simulations.head()}")
    
# --- Cálculo de la media ---
# 1. Identificar las columnas de intensidad usando el prefijo común
intensity_columns = [col for col in df_all_simulations.columns if col.startswith("Intensidad_Sim_")]

# 2. Calcular la media de esas columnas fila por fila (para cada instante de tiempo)
df_all_simulations["Intensidad_Media [A]"] = df_all_simulations[intensity_columns].mean(axis=1)

print("\n--- DataFrame Final con todas las intensidades y la Media ---")


ruta_archivo_set = os.getcwd() + '/Datos_Experimentales/Ciclos_Experimentales/Cycle_p_1000.txt'
ruta_archivo_reset = os.getcwd() + '/Datos_Experimentales/Ciclos_Experimentales/Cycle_n_1000.txt'

# Cargar datos experimentales
data_set = np.loadtxt(ruta_archivo_set)
data_reset = np.loadtxt(ruta_archivo_reset)

x_set = data_set[:, 0]
y_set = data_set[:, 1]

x_reset = data_reset[:, 0]
y_reset = abs(data_reset[:, 1])

path_medidas_ads_set = 'C:/Users/Usuario/Documents/GitHub/RRAM_Simulation/ADS data/Set.txt'
path_medidas_ads_reset = 'C:/Users/Usuario/Documents/GitHub/RRAM_Simulation/ADS data/Reset.txt'

# Cargar datos experimentales
data_ads_set = np.loadtxt(path_medidas_ads_set, skiprows=1)
data_ads_reset = np.loadtxt(path_medidas_ads_reset, skiprows=1)

x_ads_set = data_ads_set[:, 0]
y_ads_set = abs(data_ads_set[:, 1])
print(f"Datos reset modelo stanford: {x_ads_set.shape}, {y_ads_set.shape}")

x_ads_reset = data_ads_reset[:, 0]
y_ads_reset = abs(data_ads_reset[:, 1])
print(f"Datos reset modelo stanford: {x_ads_reset.shape}, {y_ads_reset.shape}")

V_ads = np.concatenate((x_ads_set, x_ads_reset))
I_ads = np.concatenate((y_ads_set, y_ads_reset))

V_medidas = np.concatenate((x_set, x_reset))
I_medidas = np.concatenate((y_set, y_reset))

print(V_ads.shape, I_ads.shape)

In [ ]:
# --- Carpeta destino de todas las figuras ---
figures_dir = base_path / "Results" / "Figures" / "Media_set"
figures_dir.mkdir(parents=True, exist_ok=True)

factors_reset = [0.9, 0.95, 1, 1.05, 1.1]
E_r_base = 1.034
E_r = [round(E_r_base * f, 3) for f in factors_reset]

fig, axes = plt.subplots(figsize=(12,9))

config_ax(axes)

# Configurar etiquetas y título
axes.set_xlabel(r"Voltage (\si{\volt})")  # (\si{\nano\meter^{-1}})
axes.set_ylabel(r"Intensity (\si{\ampere})")
axes.set_title(r"Intensidad Media durante el set a distintas $E_r$", pad=20)
axes.set_yscale('log')
k = 0
for col in intensity_columns:
    axes.plot(df_all_simulations["Tiempo [s]"], df_all_simulations[col],
             label=rf'$E_r = {E_r[k]}$', # Etiqueta más limpia
             alpha=0.6, linestyle='--') # Línea discontinua y semi-transparente
    k += 1
    

# # Graficar la intensidad media (más destacada)
# axes.plot(df_all_simulations["Tiempo [s]"], df_all_simulations["Intensidad_Media [A]"],
#          label="Intensidad Media",
#          linewidth=1.5) # Línea roja y más gruesa

axes.plot(x_set, y_set,
         label="Intensidad experimental",
         color="red", linewidth=2) # Línea roja y más gruesa

axes.legend() # Muestra la leyenda con los nombres de las series

plt.savefig(figures_dir / "I-V_medio_set.png", dpi=300, bbox_inches='tight')
plt.close(fig)


In [ ]:
factors_reset = [0.9, 0.95, 1, 1.05, 1.1]
E_r_base = 1.01
E_r = [round(E_r_base * f, 3) for f in factors_reset]

fig, axes = plt.subplots(figsize=(12,9))

config_ax(axes)

# Configurar etiquetas y título
axes.set_xlabel(r"Voltage (\si{\volt})")  # (\si{\nano\meter^{-1}})
axes.set_ylabel(r"Intensity (\si{\ampere})")
axes.set_title(r"Intensidad Media durante el set a distintas $E_r$", pad=20)
plt.legend() # Muestra la leyenda con los nombres de las series
k = 0
for col in intensity_columns:
    plt.plot(df_all_simulations["Tiempo [s]"], df_all_simulations[col],
             label=rf'$E_r = {E_r[k]}$', # Etiqueta más limpia
             alpha=0.6, linestyle='--') # Línea discontinua y semi-transparente
    k += 1
    
plt.yscale('log')

axes.legend() # Muestra la leyenda con los nombres de las series

plt.savefig(figures_dir / "I-V_varios_set.png", dpi=300, bbox_inches='tight')
plt.close(fig)

In [ ]:
print(x_ads_set)

fig, axes = plt.subplots(figsize=(12,9))

config_ax(axes)

# Configurar etiquetas y título
axes.set_xlabel(r"Voltage (\si{\volt})")  # (\si{\nano\meter^{-1}})
axes.set_ylabel(r"Intensity (\si{\ampere})")
axes.set_title(r"Medidas modelo de Stanford", pad=20)
plt.legend() # Muestra la leyenda con los nombres de las series

plt.plot(V_medidas, I_medidas, linewidth = 3, linestyle='--', label = 'Medidas experimentales') # Línea discontinua y semi-transparente
plt.scatter(V_ads, I_ads, s = 0.5,  color = 'red', label = 'Medidas modelo') # Línea discontinua y semi-transparente
    
plt.yscale('log')

axes.legend() # Muestra la leyenda con los nombres de las series

plt.savefig(figures_dir / "I-V_ADS.png", dpi=300, bbox_inches='tight')
plt.close(fig)